# AMD / HADS-M: public analysis-code documentation
**Public code package 1.0.2 | Based on the supplied analysis notebook 2.0.1**

English-language de novo analysis documentation for code-only research transparency. No raw data, individual records, executed outputs, embedded attachments, or synthetic participant datasets are distributed. Default execution is a data-free review pass: no dependency installation, upload or analysis is triggered.

This public adaptation preserves the supplied numerical routines but is not a byte-identical archive of the executed session. Changes are listed in CHANGELOG.md. Code inspection is possible without data; independent numerical verification requires authorized private input and the appropriate software environment. No patient-level analyses were executed while preparing this package.


## Authors and affiliations
Marta Nowak¹, Anna Maria Cybulska¹*, Elżbieta Grochans¹, Daria Schneider-Matyka¹, Mariusz Panczyk², Adrianna Królikowska¹, Kamila Rachubińska¹

¹ Department of Nursing, Faculty of Health Sciences, Pomeranian Medical University in Szczecin, Poland

² Department of Education and Research in Health Sciences, Faculty of Health Sciences, Medical University of Warsaw, Poland

**Correspondence:** Anna Maria Cybulska, anna.cybulska@pum.edu.pl

The creator list follows the investigator's confirmation. Specific software-development responsibilities are described separately below.


## Code and notebook contributions

Mariusz Panczyk directed the development of the analytical workflow, specified the analysis requirements, and executed the notebook in Google Colab. Microsoft Copilot assisted with drafting, documenting, and revising the Python code and notebook. Responsibility for reviewing the code, verifying the outputs, and approving the final analytical implementation rests with Mariusz Panczyk. Contributions to the manuscript do not automatically imply contributions to the analytical software.

In [ ]:
# Default review mode: no data are requested and no analysis is run.
RUN_ANALYSIS = False
INPUT_PATH = None
INPUT_SHEET = 'analysis'
OUTPUT_ROOT = '/content'  # Change for local Jupyter use.


## Optional environment and configuration
Only an authorized custodian should enable RUN_ANALYSIS and configure private input. Compatible dependency ranges are not the exact historical environment. Each optional execution records exact versions. The public input schema uses the original analysis field names and a private Boolean visual-conflict flag instead of source identifiers.


In [ ]:
if RUN_ANALYSIS:
    # Install dependencies before importing scientific packages.
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'numpy>=1.26,<3', 'pandas>=2.2,<3', 'scipy>=1.13,<2',
        'statsmodels>=0.14.4,<0.15', 'matplotlib>=3.8,<4',
        'openpyxl>=3.1,<4'])


In [ ]:
if RUN_ANALYSIS:
    from pathlib import Path
    from datetime import datetime, timezone
    import json, hashlib, zipfile, platform, importlib.metadata, warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    import statsmodels.api as sm
    from statsmodels.stats.multitest import multipletests
    from statsmodels.stats.contingency_tables import SquareTable
    from statsmodels.stats.diagnostic import het_breuschpagan
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    import matplotlib.pyplot as plt

    REFERENCE_NAME = 'Restricted analysis-ready workbook; not distributed'
    EXPECTED_SHA256 = None  # Optional: paste the approved file hash, not a different dataset hash.
    SHEET = INPUT_SHEET
    SEED = 20261022
    RESAMPLES = 10000
    TOL = 1e-9
    EYE_VISIT = {1: 1, 2: 3}
    SCALES = {'Anxiety':'LĘK', 'Depression':'DEPRESJA', 'Irritability':'AGRESJA', 'Total':'OGÓLNY'}
    RUN_ADDITIONAL_MODELS = True
    DOWNLOAD_RESULTS = False
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    OUT = Path(OUTPUT_ROOT) / ('AMD_HADSM_results_' + stamp)
    OUT.mkdir(parents=True, exist_ok=False)
    TABLES = {}; EVENTS = []; COEFS = []; MODEL_STATS = []; VIFS = []

    def save_table(name, records):
        frame = pd.DataFrame(records)
        TABLES[name] = frame.copy()
        frame.to_csv(OUT / (name + '.csv'), index=False, encoding='utf-8-sig')
        return frame

    def describe(x):
        x = pd.to_numeric(x, errors='coerce'); q = x.quantile([.25,.5,.75])
        return dict(n=int(x.count()), missing=int(x.isna().sum()), mean=x.mean(), sd=x.std(ddof=1),
            median=q.loc[.5], q1=q.loc[.25], q3=q.loc[.75], iqr=q.loc[.75]-q.loc[.25],
            minimum=x.min(), maximum=x.max())

    def bootstrap_ci(x, statistic=np.mean):
        x = np.asarray(x, float); x = x[np.isfinite(x)]
        if len(x) < 2: return (np.nan, np.nan)
        rng = np.random.default_rng(SEED); estimates = []
        for start in range(0, RESAMPLES, 250):
            samples = x[rng.integers(0,len(x),size=(min(250,RESAMPLES-start),len(x)))]
            estimates.extend(np.apply_along_axis(statistic,1,samples))
        return tuple(np.quantile(estimates,[.025,.975]))

    def adjust_holm(frame, pcol, family):
        frame = frame.copy(); frame[pcol+'_holm'] = np.nan
        for _, part in frame.groupby(family, dropna=False):
            ix = part.index[np.isfinite(pd.to_numeric(part[pcol],errors='coerce'))]
            if len(ix): frame.loc[ix,pcol+'_holm'] = multipletests(frame.loc[ix,pcol],method='holm')[1]
        return frame



## Optional input validation
See DATA_SCHEMA.csv: it is a dictionary of variables, not participant data. No upload widget is included. Private input must contain 121 rows and two flagged visual-conflict records. Do not publish the mapping of those flags to participants. Preserve row order for exact comparison of the archived resampling procedure. Source participant identifiers and item-level questionnaire fields are not needed by this public adaptation.


In [ ]:
if RUN_ANALYSIS:
    # Optional authorized execution only; no uploader or data download is provided.
    if INPUT_PATH is None:
        raise ValueError('Set INPUT_PATH to an authorized private workbook before enabling analysis.')
    INPUT = Path(INPUT_PATH)
    if not INPUT.is_file(): raise FileNotFoundError('Private input workbook not found.')
    input_hash = hashlib.sha256(INPUT.read_bytes()).hexdigest()
    if EXPECTED_SHA256 and input_hash != EXPECTED_SHA256: raise ValueError('Input SHA-256 mismatch.')
    raw = pd.read_excel(INPUT, sheet_name=SHEET, engine='openpyxl')
    raw.columns = raw.columns.astype(str).str.strip()
    if not raw.columns.is_unique: raise ValueError('Duplicate column names.')
    if len(raw)!=121: raise ValueError('The documented manuscript cohort contains 121 participants.')
    required=['Wiek','Płeć','visual_conflict']
    for token in SCALES.values(): required += [f'HADS - {token}_{t}' for t in [1,2]]
    for v in [1,2,3]:
        required += [f'{v}_VIVUS {typ} {scale}' for typ in ['SC','CC'] for scale in ['logMAR','DECIMAL']]
        required += [f'CRT_{v}',f'POCZUCIE KONTRASTU_{v}']
    required += ['TEST AMSLERA_1','TEST AMSLERA_3']
    missing=sorted(set(required)-set(raw))
    if missing: raise KeyError('Missing required fields: '+str(missing))
    flag_text=raw['visual_conflict'].astype(str).str.lower().str.strip()
    if not flag_text.isin(['true','false','1','0']).all(): raise ValueError('visual_conflict must be complete Boolean data.')
    flag=flag_text.isin(['true','1'])
    if int(flag.sum())!=2: raise ValueError('The documented exclusion sensitivity uses two flagged records.')
    # Select only documented analysis fields. Do not read source participant IDs.
    selected=[c for c in required if c!='visual_conflict']
    selected += [c for c in raw if any(c.startswith(f'HADS - {token}_{t}') and 'sten' in c.lower() for token in SCALES.values() for t in [1,2])]
    df=raw[selected].copy()
    del raw
    # Ephemeral row key for compatibility with the original regression helper; never exported.
    df['ID']=np.arange(len(df))
    for c in df:
        if c in ['Płeć','TEST AMSLERA_1','TEST AMSLERA_3','ID']: continue
        x=pd.to_numeric(df[c],errors='coerce')
        if (df[c].notna() & x.isna()).any(): raise ValueError('Non-numeric field: '+c)
        if np.isinf(x.dropna()).any(): raise ValueError('Infinite field: '+c)
        df[c]=x
    if not df['Płeć'].dropna().isin(['K','M']).all(): raise ValueError('Unexpected sex coding; see DATA_SCHEMA.csv.')
    for outcome,token in SCALES.items():
        upper=6 if outcome=='Irritability' else 48 if outcome=='Total' else 21
        for t in [1,2]:
            x=df[f'HADS - {token}_{t}']
            if ((x<0)|(x>upper)|((x%1!=0)&x.notna())).any(): raise ValueError('Invalid HADS-M score.')
    # Scores are supplied by the data custodian; no questionnaire item key is redistributed.
    for t in [1,2]:
        components=df[[f'HADS - {SCALES[o]}_{t}' for o in ['Anxiety','Depression','Irritability']]]
        total=df[f'HADS - OGÓLNY_{t}']; valid=components.notna().all(axis=1)&total.notna()
        if not np.allclose(components.loc[valid].sum(axis=1),total.loc[valid]):
            raise ValueError('Total score differs from the sum of the three component scores.')
    save_table('00_missingness',[dict(variable=c,n=len(df),missing=int(df[c].isna().sum())) for c in df if c!='ID'])
    EVENTS.append(dict(event='input_validated',n=len(df),visual_conflicts=int(flag.sum())))


## Descriptive summaries and changes
HADS assessment 1 corresponds to eye visit 1; HADS assessment 2 corresponds to eye visit 3 after seven injections. Eye visit 2 is intermediate. Amsler exists only at visits 1 and 3. No imputation or scale conversion is performed. Changes are later minus earlier; CRT reduction is earlier minus later. Direction labels are arithmetic, not clinical response thresholds.


In [ ]:
if RUN_ANALYSIS:
    derived = []; legacy_disagreements = []
    for earlier,later in [(1,2),(2,3),(1,3)]:
        for prefix,c1,c2,lower_better in [
            ('CRT',f'CRT_{earlier}',f'CRT_{later}',True),
            ('SC_logMAR',f'{earlier}_VIVUS SC logMAR',f'{later}_VIVUS SC logMAR',True),
            ('CC_logMAR',f'{earlier}_VIVUS CC logMAR',f'{later}_VIVUS CC logMAR',True),
            ('SC_decimal',f'{earlier}_VIVUS SC DECIMAL',f'{later}_VIVUS SC DECIMAL',False),
            ('CC_decimal',f'{earlier}_VIVUS CC DECIMAL',f'{later}_VIVUS CC DECIMAL',False),
            ('contrast',f'POCZUCIE KONTRASTU_{earlier}',f'POCZUCIE KONTRASTU_{later}',False)]:
            delta = df[c2]-df[c1]
            name = f'analysis_{prefix}_delta_{later}_minus_{earlier}'
            df[name] = delta
            labels = pd.Series('missing',index=df.index)
            labels.loc[delta.notna() & (abs(delta)<=TOL)] = 'no change'
            labels.loc[delta.notna() & (abs(delta)>TOL)] = np.where(
                delta.loc[delta.notna() & (abs(delta)>TOL)] < 0 if lower_better else delta.loc[delta.notna() & (abs(delta)>TOL)] > 0,
                'improvement','deterioration')
            df[name+'_label'] = labels
            if prefix=='CRT': df[f'analysis_CRT_reduction_{earlier}_to_{later}'] = -delta
            derived.append(dict(variable=name,definition=c2+' minus '+c1,lower_is_better=lower_better))
    save_table('01_derived_dictionary',derived)
    num=[]; cat=[]
    for c in ['Wiek'] + [f'HADS - {s}_{t}' for s in SCALES.values() for t in [1,2]] + [f'{v}_VIVUS {typ} logMAR' for v in [1,2,3] for typ in ['SC','CC']] + [f'CRT_{v}' for v in [1,2,3]] + [f'POCZUCIE KONTRASTU_{v}' for v in [1,2,3]]:
        num.append(dict(variable=c,**describe(df[c])))
    for c in ['Płeć', 'TEST AMSLERA_1', 'TEST AMSLERA_3']:
        for value,n in df[c].fillna('[missing]').value_counts().items(): cat.append(dict(variable=c,category=str(value),n=int(n),percent=100*n/len(df)))
    save_table('02_descriptive_numeric',num); save_table('02_descriptive_categorical',cat)
    save_table('02_HADS_descriptive',[dict(outcome=l,time=t,**describe(df[f'HADS - {s}_{t}'])) for l,s in SCALES.items() for t in [1,2]])


## Paired quantitative outcomes
Anxiety and depression form the primary family; irritability and total form the secondary family. The two-item legacy AGRESJA domain is labelled Irritability in manuscript-facing outputs. Paired t tests address mean changes; Wilcoxon and sign-flip procedures are sensitivity analyses with different assumptions. Percentile bootstrap CIs use 10,000 paired-difference resamples. Monte Carlo sign-flip tests use 10,000 draws and the plus-one p-value correction. Seed 20261022 is reset for each procedure as in the supplied source. Cohen's dz uses the SD of paired differences; rank-biserial correlation is signed. Holm adjustment is separate by outcome family and test procedure; CIs are pointwise. This hierarchy is not presented as evidence of preregistration.


In [ ]:
if RUN_ANALYSIS:
    paired=[]
    for outcome,token in SCALES.items():
        a=df[[f'HADS - {token}_1',f'HADS - {token}_2']].dropna()
        delta=(a.iloc[:,1]-a.iloc[:,0]).to_numpy(float); n=len(delta)
        if n<3: raise ValueError('Insufficient paired observations: '+outcome)
        mean=delta.mean(); sd=delta.std(ddof=1); ci=bootstrap_ci(delta); mci=bootstrap_ci(delta,np.median)
        tt=stats.ttest_rel(a.iloc[:,1],a.iloc[:,0]) if sd>0 else None
        wt=stats.wilcoxon(delta,zero_method='wilcox',method='auto') if np.any(delta!=0) else None
        rng=np.random.default_rng(SEED); exceed=0
        for start in range(0,RESAMPLES,250):
            signs=rng.choice([-1,1],size=(min(250,RESAMPLES-start),n))
            exceed+=int((abs((signs*delta).mean(axis=1))>=abs(mean)-1e-12).sum())
        nz=delta[delta!=0]; ranks=stats.rankdata(abs(nz))
        rbc=(ranks[nz>0].sum()-ranks[nz<0].sum())/ranks.sum() if len(nz) else 0
        half=stats.t.ppf(.975,n-1)*sd/np.sqrt(n)
        paired.append(dict(outcome=outcome,family='primary' if outcome in ['Anxiety','Depression'] else 'secondary',n=n,
            mean_change=mean,sd_change=sd,ci_low=ci[0],ci_high=ci[1],t_ci_low=mean-half,t_ci_high=mean+half,
            median_change=np.median(delta),median_ci_low=mci[0],median_ci_high=mci[1],
            cohen_dz=mean/sd if sd>0 else np.nan,rank_biserial=rbc,t=tt.statistic if tt else np.nan,df=n-1,
            p_t=tt.pvalue if tt else (1. if mean==0 else np.nan),W=wt.statistic if wt else 0.,p_wilcoxon=wt.pvalue if wt else 1.,p_signflip=(exceed+1)/(RESAMPLES+1)))
    paired=pd.DataFrame(paired)
    for p in ['p_t','p_wilcoxon','p_signflip']: paired=adjust_holm(paired,p,'family')
    save_table('03_HADS_paired',paired)


## Exploratory stored categories
C1–C3 are stored nominal codes, not verified clinical thresholds. Only anxiety, depression and irritability categorical results are reported. Total is a quantitative outcome only.

For the archived irritability adjusted category p value, the original secondary family included an omitted total-code test. That nuisance calculation is retained internally before all total categorical outputs are discarded. This does not validate a total-score classification. If total codes are unavailable, the archived irritability adjusted p value is withheld, rather than changing its family silently. Stuart–Maxwell tests are asymptotic and require caution with sparse transitions.


In [ ]:
if RUN_ANALYSIS:
    # Stored screening categories: do not infer new clinical thresholds.
    category_rows=[]; transitions=[]; marginal=[]
    for outcome,token in SCALES.items():
        candidates=[[c for c in df if c.startswith(f'HADS - {token}_{t}') and 'sten' in c.lower()] for t in [1,2]]
        if any(len(c)!=1 for c in candidates):
            EVENTS.append(dict(event='categories_unavailable',outcome=outcome));continue
        cols=[c[0] for c in candidates]; a=df[cols].dropna()
        if not a.isin([1,2,3]).all().all(): raise ValueError('Unexpected category codes: '+outcome)
        tab=pd.crosstab(a.iloc[:,0],a.iloc[:,1]).reindex(index=[1,2,3],columns=[1,2,3],fill_value=0)
        for t,c in zip([1,2],cols):
            den=int(df[c].notna().sum())
            for k in [1,2,3]: category_rows.append(dict(outcome=outcome,time=t,category=k,n=int((df[c]==k).sum()),valid_n=den,percent=100*(df[c]==k).sum()/den if den else np.nan))
        for i in [1,2,3]:
            for j in [1,2,3]: transitions.append(dict(outcome=outcome,baseline=i,followup=j,n=int(tab.loc[i,j])))
        try:
            arr=tab.to_numpy(); active=(arr.sum(0)+arr.sum(1))>0; arr=arr[np.ix_(active,active)]
            if len(arr)<2: raise ValueError('No category variation')
            res=SquareTable(arr,shift_zeros=False).homogeneity(method='stuart_maxwell')
            statistic,p,deg=res.statistic,res.pvalue,res.df; status='asymptotic; inspect sparse transitions'
        except Exception as e: statistic,p,deg=np.nan,np.nan,np.nan; status=str(e)
        marginal.append(dict(outcome=outcome,family='primary' if outcome in ['Anxiety','Depression'] else 'secondary',n=len(a),chi2=statistic,df=deg,p=p,status=status))
    # Keep the historical two-test secondary adjustment before discarding total categories.
    # This is an archival calculation, not a validated clinical total-score classification.
    category_rows=[r for r in category_rows if r['outcome']!='Total']
    transitions=[r for r in transitions if r['outcome']!='Total']
    save_table('04_categories',category_rows);save_table('04_transitions',transitions)
    if marginal:
        archived=adjust_holm(pd.DataFrame(marginal),'p','family')
        total_available=any(r['outcome']=='Total' for r in marginal)
        archived=archived.loc[archived.outcome!='Total'].copy()
        if not total_available:
            archived.loc[archived.outcome=='Irritability','p_holm']=np.nan
            EVENTS.append(dict(event='archived_category_adjustment_unavailable',reason='No total-code test; irritability adjusted p withheld rather than silently changing its family.'))
        archived['adjustment_note']=np.where(archived.outcome=='Irritability',
            'Archived irritability/omitted-total family; total categorical outputs excluded',
            'Anxiety/depression family')
        save_table('04_marginal_homogeneity',archived)



## Regression models and diagnostics
Core models include uncorrected logMAR acuity, corrected logMAR acuity and contrast at the corresponding visit. HC3 uses t-based inference; conventional OLS estimates are also retained. Standardized coefficients condition on observed sample scaling.

Six separate coefficient families: core; exclusion of privately flagged visual records; decimal-acuity sensitivity; baseline-adjusted final scores; unadjusted Amsler; adjusted final Amsler. Each documented family has 24 non-intercept coefficients with a separate Holm correction. Intercepts and overall model F tests are unadjusted. Baseline-adjusted final-score models include the baseline same-scale score, baseline ophthalmic predictors, age and sex. Adjusted Amsler models include final Amsler, baseline same-scale score, age and sex. Amsler is nominal; no metamorphopsia is the literal reference category, not a claim of a normal test.

Diagnostics include Breusch–Pagan, residual Shapiro–Wilk, VIF, unscaled condition number and counts of Cook's D > 4/n. No automatic influence deletion or predictor selection is performed. Individual diagnostic exports and scatterplots are removed. These observational models do not estimate a causal treatment effect.


In [ ]:
if RUN_ANALYSIS:
    def fit_model(data,ycol,xcols,name,family,standardize=True):
        a=data[['ID',ycol]+xcols].copy()
        for c in [ycol]+xcols:a[c]=pd.to_numeric(a[c],errors='coerce')
        a=a.replace([np.inf,-np.inf],np.nan).dropna()
        if len(a)<=len(xcols)+5 or a[ycol].nunique()<2 or any(a[c].nunique()<2 for c in xcols):
            EVENTS.append(dict(event='model_skipped',model=name,reason='sample size or constant variable'));return
        X=sm.add_constant(a[xcols],has_constant='add'); y=a[ycol]
        if np.linalg.matrix_rank(X)<X.shape[1]:
            EVENTS.append(dict(event='model_skipped',model=name,reason='rank deficiency'));return
        fit=sm.OLS(y,X).fit(); robust=fit.get_robustcov_results(cov_type='HC3',use_t=True)
        ci=np.asarray(robust.conf_int()); ci0=np.asarray(fit.conf_int())
        for j,c in enumerate(X):
            scale=a[c].std()/y.std() if c!='const' and standardize else np.nan
            COEFS.append(dict(model=name,family=family,predictor=c,n=len(a),b=fit.params.iloc[j],se_OLS=fit.bse.iloc[j],p_OLS=fit.pvalues.iloc[j],ci_OLS_low=ci0[j,0],ci_OLS_high=ci0[j,1],se_HC3=robust.bse[j],t_HC3=robust.tvalues[j],p_HC3=robust.pvalues[j],ci_low=ci[j,0],ci_high=ci[j,1],beta=fit.params.iloc[j]*scale,beta_ci_low=ci[j,0]*scale,beta_ci_high=ci[j,1]*scale))
        influence=fit.get_influence(); cooks=influence.cooks_distance[0]
        bp=het_breuschpagan(fit.resid,X)
        MODEL_STATS.append(dict(model=name,family=family,n=len(a),r2=fit.rsquared,adjusted_r2=fit.rsquared_adj,F_OLS=fit.fvalue,p_F_OLS=fit.f_pvalue,F_HC3=float(np.asarray(robust.fvalue).squeeze()),p_F_HC3=float(np.asarray(robust.f_pvalue).squeeze()),df_residual=fit.df_resid,p_breusch_pagan=bp[1],p_shapiro_residuals=stats.shapiro(fit.resid).pvalue,cooks_above_4_n=int((cooks>4/len(a)).sum()),condition_number=fit.condition_number))
        for j,c in enumerate(X):
            if c!='const':VIFS.append(dict(model=name,predictor=c,VIF=variance_inflation_factor(X.to_numpy(),j)))

    for outcome,token in SCALES.items():
        for t in [1,2]:
            v=EYE_VISIT[t]; y=f'HADS - {token}_{t}';x=[f'{v}_VIVUS SC logMAR',f'{v}_VIVUS CC logMAR',f'POCZUCIE KONTRASTU_{v}']
            fit_model(df,y,x,f'core_{outcome}_T{t}','core')
            fit_model(df.loc[~flag],y,x,f'no_visual_conflict_{outcome}_T{t}','sensitivity_flag')
            fit_model(df,y,[f'{v}_VIVUS SC DECIMAL',f'{v}_VIVUS CC DECIMAL',f'POCZUCIE KONTRASTU_{v}'],f'decimal_{outcome}_T{t}','sensitivity_decimal')
        if RUN_ADDITIONAL_MODELS:
            a=df.copy();a['sex_M']=a['Płeć'].map({'K':0,'M':1})
            fit_model(a,f'HADS - {token}_2',[f'HADS - {token}_1','1_VIVUS SC logMAR','1_VIVUS CC logMAR','POCZUCIE KONTRASTU_1','Wiek','sex_M'],f'baseline_adjusted_{outcome}','baseline_adjusted')

    levels=['BRAK METAMORFOPSJI','OBECNOŚĆ METAMORFOPSJI','OBECNOŚĆ MROCZKA','OBECNOŚĆ MROCZKA I METAMORFOPSJI']
    amsler_counts=[]
    for t in [1,2]:
        v=EYE_VISIT[t]; c=f'TEST AMSLERA_{v}'
        if set(df[c].dropna())-set(levels):raise ValueError('Unknown Amsler category.')
        a=df.copy(); dummy=pd.get_dummies(pd.Categorical(a[c],categories=levels),prefix='Amsler',drop_first=True,dtype=float);dummy.index=a.index;dummy.loc[a[c].isna(),:]=np.nan
        a=pd.concat([a,dummy],axis=1)
        for level in levels:amsler_counts.append(dict(time=t,eye_visit=v,category=level,n=int((df[c]==level).sum())))
        for outcome,token in SCALES.items():
            fit_model(a,f'HADS - {token}_{t}',list(dummy),f'Amsler_{outcome}_T{t}','Amsler',False)
            if t==2 and RUN_ADDITIONAL_MODELS:
                a['sex_M']=a['Płeć'].map({'K':0,'M':1})
                fit_model(a,f'HADS - {token}_2',list(dummy)+[f'HADS - {token}_1','Wiek','sex_M'],f'Amsler_adjusted_{outcome}','Amsler_adjusted',False)
    save_table('05_Amsler_counts',amsler_counts)
    reg=pd.DataFrame(COEFS)
    if reg.empty:raise ValueError('No models could be fitted; inspect the event log.')
    reg['p_HC3_holm']=np.nan
    for family,part in reg[reg.predictor!='const'].groupby('family'):
        ix=part.index[np.isfinite(part.p_HC3)]
        if len(ix):reg.loc[ix,'p_HC3_holm']=multipletests(reg.loc[ix,'p_HC3'],method='holm')[1]
    save_table('06_model_coefficients',reg);save_table('06_model_summaries',MODEL_STATS);save_table('06_VIF',VIFS)


    counts=reg.loc[reg.predictor!='const'].groupby('family').size()
    save_table('06_multiplicity_audit',counts.rename('non_intercept_coefficients').reset_index())
    if len(counts)!=6 or not (counts==24).all():
        EVENTS.append(dict(event='model_family_size_mismatch',reason='Inspect skipped models; manuscript families have 24 coefficients each.'))


## Optional aggregate figures

No total categorical panel is generated. Figures contain aggregate estimates only and are not precomputed or embedded in the public notebook.

In [ ]:
if RUN_ANALYSIS:
    # Figures use aggregate estimates only; no patient identifiers.
    def export_figure(fig,name):
        fig.tight_layout()
        for ext in ['png','tiff','pdf']:fig.savefig(OUT/(name+'.'+ext),dpi=600,bbox_inches='tight')
        plt.close(fig)
    fig,ax=plt.subplots(figsize=(7,4.5))
    for i,(outcome,token) in enumerate(SCALES.items()):
        means=[];lower=[];upper=[]
        for t in [1,2]:
            x=df[f'HADS - {token}_{t}'].dropna().to_numpy(float);m=x.mean();ci=bootstrap_ci(x)
            means.append(m);lower.append(max(0,m-ci[0]));upper.append(max(0,ci[1]-m))
        ax.errorbar(np.array([1,2])+(i-1.5)*.035,means,yerr=[lower,upper],marker='o',capsize=4,label=outcome)
    ax.set(xticks=[1,2],xticklabels=['Baseline','Final assessment'],ylabel='HADS-M score: mean and 95% bootstrap CI');ax.legend(frameon=False);ax.spines[['top','right']].set_visible(False)
    export_figure(fig,'Figure_1_HADS_scores')
    fig,ax=plt.subplots(figsize=(7,4.5))
    ax.errorbar(paired.mean_change,np.arange(len(paired)),xerr=[np.maximum(0,paired.mean_change-paired.ci_low),np.maximum(0,paired.ci_high-paired.mean_change)],fmt='o',capsize=4)
    ax.axvline(0,color='black',lw=.8);ax.set(yticks=np.arange(len(paired)),yticklabels=paired.outcome,xlabel='Final minus baseline score; 95% bootstrap CI');ax.spines[['top','right']].set_visible(False)
    export_figure(fig,'Figure_2_HADS_changes')
    if category_rows:
        fig,axes=plt.subplots(1,3,figsize=(10,4));ct=TABLES['04_categories']
        for ax,outcome in zip(axes,[o for o in SCALES if o!='Total']):
            bottom=np.zeros(2)
            for k in [1,2,3]:
                vals=ct[(ct.outcome==outcome)&(ct.category==k)].sort_values('time').percent.to_numpy()
                if len(vals)!=2:continue
                ax.bar([1,2],vals,bottom=bottom,label=f'C{k}');bottom+=vals
            ax.set(title=outcome,xticks=[1,2],xticklabels=['Baseline','Final'],ylim=(0,100),ylabel='Percentage')
        handles,labels=axes[-1].get_legend_handles_labels();fig.legend(handles,labels,loc='upper center',ncol=3,bbox_to_anchor=(.5,1.06),frameon=False)
        export_figure(fig,'Figure_3_HADS_categories')


## Optional private aggregate export

No automatic publication, repository mutation or patient-level export occurs. Output table names are technical; MANUSCRIPT_MAP.md provides the agreed manuscript crosswalk, not a claim of identical DOCX formatting.

In [ ]:
if RUN_ANALYSIS:
    # Private aggregate export only; never include the input workbook or row-level diagnostics.
    versions={p:importlib.metadata.version(p) for p in ['numpy','pandas','scipy','statsmodels','matplotlib','openpyxl']}
    manifest=dict(public_code_version='1.0.2',source_analysis_version='2.0.1',analysis='de novo',
        input_sha256=input_hash,n=len(df),seed=SEED,resamples=RESAMPLES,eye_visit=EYE_VISIT,
        software=versions,python=platform.python_version(),raw_data_included=False,individual_outputs_included=False)
    (OUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    (OUT/'events.json').write_text(json.dumps(EVENTS,ensure_ascii=False,indent=2),encoding='utf-8')
    (OUT/'requirements_executed.txt').write_text(chr(10).join(f'{p}=={v}' for p,v in versions.items()),encoding='utf-8')
    with pd.ExcelWriter(OUT/'aggregate_tables.xlsx',engine='openpyxl') as writer:
        for i,(name,frame) in enumerate(TABLES.items()):frame.to_excel(writer,sheet_name=(str(i)+'_'+name)[:31],index=False)
    (OUT/'checksums.json').write_text(json.dumps({p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.iterdir() if p.is_file()},indent=2),encoding='utf-8')
    archive=Path(str(OUT)+'.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUT.iterdir()):
            if p.is_file():z.write(p,p.name)
    with zipfile.ZipFile(archive) as z:assert z.testzip() is None
    print('Optional private aggregate export complete. Review small cells before public release.')


In [ ]:
print('Review mode complete: no data requested and no analyses executed.' if not RUN_ANALYSIS else 'Optional authorized analysis completed.')

## Validation and release status
Python syntax, notebook structure and default review execution were checked. No data-mode execution or new statistical analysis was performed. Numerical identity to the manuscript was not re-established by packaging. Before release, confirm final table numbering, the exact original software environment, creator metadata and the software license. LICENSE is explicitly pending, not a license grant.


## License

The original code and accompanying documentation are licensed under **CC-BY-4.0**. See `LICENSE` for the notice and official legal terms. Patient data are not included or licensed. Third-party materials retain their own rights. This is not a claim that Frontiers requires this software license.